# 🏭 ENM412 – MAN Türkiye A.Ş. Stok Yönetimi Modernizasyonu
**Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN** | ENM412 | Gazi Üniversitesi

---

## Pipeline Özeti

| Adım | İçerik |
|------|--------|
| 1 | Veri yükleme + Talep analizi + Segmentasyon |
| 2 | ML eğitimi: RF · XGBoost · LightGBM · CatBoost (Optuna) |
| 3 | Klasik yöntemler: Hareketli Ort. · Üstel · Croston-SBA |
| 4 | Metrik: MAE · RMSE · WAPE · sMAPE → Voting şampiyon |
| 5 | Stok optimizasyonu: Grid Search + Optuna + SimPy |
| 6 | Karşılaştırma: EOQ vs ML+SimPy |

**Segment Mantığı:**
- `is_intermittent=1` → DUZENSIZ → Croston-SBA öncelikli
- `is_low_volume=1` → DUSUK → Min-Max politika
- Diğer → DUZENLI → ML yarışması

---

## 1️⃣ Kurulum

In [ ]:
!pip install optuna xgboost lightgbm catboost simpy plotly openpyxl scikit-learn scipy -q
print('✅ Kurulum tamamlandı')

## 2️⃣ Dosya Yükleme

In [ ]:
from google.colab import files
uploaded = files.upload()  # tüketim.xlsx + .py dosyaları
for f in uploaded: print(f'  ✅ {f}')

In [ ]:
import os, sys
sys.path.insert(0, '.')
gerekli = ['tüketim.xlsx','veri_hazirlama.py','evaluate.py',
           'model_egitim.py','feature_engineering.py','main.py']
for f in gerekli:
    print(f'  {"✅" if os.path.exists(f) else "❌ EKSİK"} {f}')

## 3️⃣ Veri Analizi ve Segmentasyon

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from veri_hazirlama import veri_yukle, talep_analizi

veri = veri_yukle('tüketim.xlsx')
analiz_df = talep_analizi(veri['ml_df'])
print(analiz_df.head(10).to_string(index=False))

In [ ]:
# Talep dağılımı görseli
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Segment dağılımı
seg_counts = analiz_df['Segment'].value_counts()
axes[0].bar(seg_counts.index, seg_counts.values,
            color=['#1E4D8C','#C8102E','#F39200'])
axes[0].set_title('Talep Segment Dağılımı', fontweight='bold')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Parça Sayısı')
for i, v in enumerate(seg_counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')

# 2. Sıfır talep oranı dağılımı
axes[1].hist(analiz_df['Sifir_Oran'], bins=20, color='#1E4D8C', edgecolor='white')
axes[1].set_title('Sıfır Talep Oranı Dağılımı', fontweight='bold')
axes[1].set_xlabel('Sıfır Oranı')
axes[1].set_ylabel('Parça Sayısı')
axes[1].axvline(0.33, color='#C8102E', linestyle='--', label='Aralıklı Eşiği')
axes[1].legend()

# 3. Ortalama talep dağılımı (log ölçek)
talep_pos = analiz_df[analiz_df['Ort_Talep'] > 0]['Ort_Talep']
axes[2].hist(np.log1p(talep_pos), bins=30, color='#00843D', edgecolor='white')
axes[2].set_title('Log(Ort. Talep) Dağılımı', fontweight='bold')
axes[2].set_xlabel('log(1 + Ort. Talep)')
axes[2].set_ylabel('Parça Sayısı')

plt.tight_layout()
plt.savefig('talep_analizi.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik kaydedildi: talep_analizi.png')

## 4️⃣ Model Eğitimi

In [ ]:
from model_egitim import global_ml_egit

# Tam eğitim
global_ml = global_ml_egit(veri['train_df'], n_trials=30)

# Hızlı test için:
# global_ml = global_ml_egit(veri['train_df'], n_trials=10)

## 5️⃣ Tüm Parçalar İçin 7 Yöntem Karşılaştırması

In [ ]:
from model_egitim import batch_tahmin

batch_df = batch_tahmin(veri['ml_df'], global_ml, veri['parcalar'])
print(f'✅ {len(batch_df):,} parça tamamlandı')
print()
print('Şampiyon Dağılımı:')
print(batch_df['Sampiyon'].value_counts().to_string())

In [ ]:
# Metrik özeti
print('Ortalama Metrikler (şampiyon modeller, medyan):')
print(batch_df[['MAE','RMSE','WAPE','sMAPE']].median().round(2).to_string())

# Şampiyon dağılımı grafiği
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

samp_counts = batch_df['Sampiyon'].value_counts()
colors = ['#1E4D8C','#C8102E','#00843D','#F39200','#888888','#AAAAAA','#555555']
axes[0].barh(samp_counts.index, samp_counts.values, color=colors[:len(samp_counts)])
axes[0].set_title('Şampiyon Model Dağılımı (Voting)', fontweight='bold')
axes[0].set_xlabel('Parça Sayısı')
for i, v in enumerate(samp_counts.values):
    axes[0].text(v+1, i, str(v), va='center')

# WAPE dağılımı
wape_clean = batch_df['WAPE'].dropna()
axes[1].hist(wape_clean, bins=30, color='#1E4D8C', edgecolor='white')
axes[1].set_title('WAPE Dağılımı (Tüm Parçalar)', fontweight='bold')
axes[1].set_xlabel('WAPE (%)')
axes[1].set_ylabel('Parça Sayısı')
axes[1].axvline(wape_clean.median(), color='#C8102E', linestyle='--',
                label=f'Medyan: {wape_clean.median():.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ Tekil Ürün Analizi

In [ ]:
from model_egitim    import parca_tahmin
from feature_engineering import parca_optimize, aksiyon_uyarisi
from evaluate import metrik_tablosu

PID = 'Part-5'  # İstediğin parçayı seç

# Tahmin
t = parca_tahmin(PID, veri['ml_df'], global_ml, n_ay=6)

# 7 yöntem metrik tablosu
tablo = metrik_tablosu(t['tum_met'], t['sampiyon'])
print(f'=== {PID} — Segment: {t["segment"]} | ABC:{t["abc"]} XYZ:{t["xyz"]} ===')
print(tablo.to_string(index=False))
print(f'\nŞampiyon: {t["sampiyon"]} ({t["sampiyon_tip"]})')
print(f'Gelecek 6 ay tahmini: {[round(x,1) for x in t["tahminler"]]}')

In [ ]:
# Tahmin grafiği
ETIKET_TRAIN = ['Oca-22','Şub-22','Mar-22','Nis-22','May-22','Haz-22',
                 'Tem-22','Ağu-22','Eyl-22','Eki-22','Kas-22','Ara-22',
                 'Oca-23','Şub-23','Mar-23','Nis-23','May-23','Haz-23',
                 'Tem-23','Ağu-23','Eyl-23','Eki-23','Kas-23','Ara-23',
                 'Oca-24','Şub-24','Mar-24','Nis-24','May-24','Haz-24']
ETIKET_TEST  = ['Tem-24','Ağu-24','Eyl-24','Eki-24','Kas-24','Ara-24']
ETIKET_GEL   = ['Oca-25','Şub-25','Mar-25','Nis-25','May-25','Haz-25']
RENK = {'RF':'#1E4D8C','XGBoost':'#C8102E','LightGBM':'#00843D','CatBoost':'#F39200',
        'Hareketli Ort.':'#888','Üstel Düzeltme':'#AAA','Croston-SBA':'#555'}

fig, ax = plt.subplots(figsize=(14, 5))

ts_train = t['ts_train']
y_test   = t['y_test_ham']
tahminler = t['tahminler']

# Eğitim
ax.plot(ETIKET_TRAIN[:len(ts_train)], ts_train, color='#1E4D8C',
        linewidth=2, label='Gerçek (Eğitim)')

# Test gerçek
ax.plot(ETIKET_TEST[:len(y_test)], y_test, color='#1E4D8C',
        linewidth=2.5, linestyle='--', marker='o', markersize=5,
        label='Gerçek (Test)')

# ML tahminleri
for m_adi, m_pred in t['tum_ml_pred'].items():
    is_s = m_adi == t['sampiyon']
    ax.plot(ETIKET_TEST[:len(m_pred)], m_pred,
            color=RENK.get(m_adi,'#999'),
            linewidth=3 if is_s else 1,
            alpha=1.0 if is_s else 0.4,
            label=f'{m_adi}{" ⭐" if is_s else ""}')

# Klasik tahminler
for g_adi, g_pred in t['tum_gel_pred'].items():
    is_s = g_adi == t['sampiyon']
    ax.plot(ETIKET_TEST[:len(g_pred)], g_pred,
            color=RENK.get(g_adi,'#888'),
            linewidth=3 if is_s else 1,
            linestyle='dashdot', alpha=1.0 if is_s else 0.35,
            label=f'{g_adi}{" ⭐" if is_s else ""}')

# Gelecek tahmin
if tahminler:
    ax.plot(ETIKET_GEL[:len(tahminler)], tahminler,
            color=RENK.get(t['sampiyon'],'#C8102E'),
            linewidth=3, linestyle='--', marker='D', markersize=8,
            label=f'Tahmin ({t["sampiyon"]}) → Gelecek 6 Ay')

ax.axvline(x='Tem-24', color='orange', linestyle=':', linewidth=1.5)
ax.axvline(x='Oca-25', color='orange', linestyle=':', linewidth=1.5)
ax.set_title(f'{PID} – 7 Yöntem Karşılaştırması (⭐ Şampiyon: {t["sampiyon"]})',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Tüketim (adet)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'tahmin_{PID}.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Stok Optimizasyonu (Grid Search + Optuna + SimPy)

In [ ]:
o = parca_optimize(
    parca_kodu    = PID,
    opt_df        = veri['opt_df'],
    abc_df        = veri['abc_df'],
    tahmin_listesi= t['tahminler'],
    segment       = t['segment'],
    grid_adim     = 12,
    n_trials      = 40,
    n_rep         = 20,
)
uy = aksiyon_uyarisi(o, t['tahminler'])

print(f'=== {PID} – Stok Politikası ({o["yontem"]}) ===')
print(f'\nML + Optuna Önerisi:')
print(f'  Q* = {o["optimal_Q"]:,} adet/sipariş')
print(f'  r* = {o["optimal_r"]:,} adet (yeniden sipariş noktası)')
print(f'  SS*= {o["optimal_SS"]:,} adet (emniyet stoğu)')
print(f'  Hizmet Düzeyi: %{o["sim_HZ"]*100:.1f} (SimPy)')
print(f'\nEOQ Klasik Referans:')
print(f'  Q  = {o["Q_eoq"]:,} | r  = {o["r_eoq"]:,} | SS = {o["SS_eoq"]:,}')
print(f'\nMaliyet Karşılaştırması:')
print(f'  EOQ Maliyet : {o["eoq_TC"]:,.2f} TL/ay')
print(f'  ML Maliyet  : {o["sim_TC"]:,.2f} TL/ay (SimPy)')
print(f'  Tasarruf    : {o["tasarruf_tl"]:,.2f} TL/ay ({o["tasarruf_oran"]:.1f}%)')
print(f'\n{uy["mesaj"]}')

In [ ]:
# Maliyet karşılaştırma grafiği
fig, ax = plt.subplots(figsize=(8, 5))

kategoriler = ['Elde Tutma', 'Sipariş', 'Stoksuz Kalma', 'TOPLAM']
eoq_vals = [o['eoq_ET'], o['eoq_SI'], o['eoq_SK'], o['eoq_TC']]
ml_vals  = [o['sim_ET'], o['sim_SI'], o['sim_SK'], o['sim_TC']]

x    = np.arange(len(kategoriler))
wd   = 0.35
b1   = ax.bar(x - wd/2, eoq_vals, wd, label='EOQ Klasik',
               color='#F39200', edgecolor='white', linewidth=1.5)
b2   = ax.bar(x + wd/2, ml_vals, wd, label='ML + Optuna + SimPy',
               color='#1E4D8C', edgecolor='white', linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(kategoriler)
ax.set_ylabel('TL/ay')
ax.set_title(f'{PID} – Maliyet Karşılaştırması', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.grid(axis='y', alpha=0.3)

for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h*1.01,
            f'{h:,.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'maliyet_{PID}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Tasarruf: {o["tasarruf_tl"]:,.2f} TL/ay (%{o["tasarruf_oran"]:.1f})')

## 8️⃣ Cache Kaydet ve İndir

In [ ]:
import pickle
sonuc = {'veri': veri, 'global_ml': global_ml,
         'batch_df': batch_df, 'analiz_df': analiz_df}
with open('enm412_cache.pkl', 'wb') as f:
    pickle.dump(sonuc, f, protocol=4)
print(f'✅ Cache: {os.path.getsize("enm412_cache.pkl")/1024/1024:.1f} MB')

In [ ]:
from google.colab import files
files.download('enm412_cache.pkl')
# Grafikleri de indir
for f in ['talep_analizi.png','model_karsilastirma.png',
          f'tahmin_{PID}.png', f'maliyet_{PID}.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'✅ {f}')

---
**ENM412 – Endüstri Mühendisliğinde Tasarım II**
Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN | MAN Türkiye A.Ş. | 2024-2025